<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Загрузка обученной модели из эксперимента mlflow

</div>

In [1]:
import os
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

os.environ['MLFLOW_HTTP_REQUEST_TIMEOUT'] = '600'
os.environ['MLFLOW_HTTP_REQUEST_MAX_RETRIES'] = '5'

tracking_uri = "http://127.0.0.1:5000"
registered_model_name = "hr-ai-scout-prd"
prd_alias = "PRD"

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)

In [2]:
prd_uri = f"models:/{registered_model_name}@{prd_alias}"
loaded_prd = mlflow.sklearn.load_model(prd_uri)

client = MlflowClient()
mv = client.get_model_version_by_alias(registered_model_name, prd_alias)
{'name': mv.name, 'alias': prd_alias, 'version': mv.version, 'run_id': mv.run_id}

{'name': 'hr-ai-scout-prd',
 'alias': 'PRD',
 'version': '3',
 'run_id': '6e64aeb2c7ef499b9763a0369edda24c'}

In [3]:
test_samples = pd.DataFrame([
    {
        'vacancy_area':                          'Москва',
        'vacancy_experience':                    'От 3 до 6 лет',
        'vacancy_employment':                    'Полная занятость',
        'vacancy_schedule':                      'Удаленная работа',
        'resume_salary':                         150000.0,
        'resume_age':                            32,
        'resume_experience_months':              96,
        'resume_location':                       'Москва',
        'resume_gender':                         'Мужчина',
        'resume_applicant_status':               'Активно ищет работу',
        'resume_last_company_experience_months': 36,
        'location_matching':                     1,
        'resume_skill_count_in_vacancy':         5,
        'last_position_in_vacancy':              0.6,
        'similarity_score_tfidf':                0.45,
        'als_score':                             0.18,
        'sim_labse_hr_finetuned':                0.72,
    },
    {
        'vacancy_area':                          'Москва',
        'vacancy_experience':                    'От 3 до 6 лет',
        'vacancy_employment':                    'Полная занятость',
        'vacancy_schedule':                      'Удаленная работа',
        'resume_salary':                         60000.0,
        'resume_age':                            55,
        'resume_experience_months':              300,
        'resume_location':                       'Самара',
        'resume_gender':                         'Женщина',
        'resume_applicant_status':               'NDT',
        'resume_last_company_experience_months': 120,
        'location_matching':                     0,
        'resume_skill_count_in_vacancy':         0,
        'last_position_in_vacancy':              0.05,
        'similarity_score_tfidf':                0.02,
        'als_score':                             0.0,
        'sim_labse_hr_finetuned':                0.18,
    },
])
test_samples

,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,resume_salary,resume_age,resume_experience_months,resume_location,resume_gender,resume_applicant_status,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,similarity_score_tfidf,als_score,sim_labse_hr_finetuned
0,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,150000.0,32,96,Москва,Мужчина,Активно ищет работу,36,1,5,0.60,0.45,0.18,0.72
1,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,60000.0,55,300,Самара,Женщина,NDT,120,0,0,0.05,0.02,0.00,0.18


In [4]:
proba = loaded_prd.predict_proba(test_samples)[:, 1]
pred  = (proba >= 0.5).astype(int)

result = test_samples[[
    'vacancy_area', 'resume_location', 'location_matching',
    'resume_experience_months', 'resume_skill_count_in_vacancy',
    'last_position_in_vacancy', 'similarity_score_tfidf',
    'als_score', 'sim_labse_hr_finetuned',
]].copy()
result['pred_proba'] = proba
result['pred'] = pred
result

,vacancy_area,resume_location,location_matching,resume_experience_months,resume_skill_count_in_vacancy,last_position_in_vacancy,similarity_score_tfidf,als_score,sim_labse_hr_finetuned,pred_proba,pred
0,Москва,Москва,1,96,5,0.60,0.45,0.18,0.72,0.999737,1
1,Москва,Самара,0,300,0,0.05,0.02,0.00,0.18,0.000130,0
